# Age Group Detection — Quantitative and Qualitative Model Comparison
## IN3060 Computer Vision Coursework

**Run this notebook AFTER training all three models.**

Loads HOG+SVM, HOG+MLP, and CNN (ResNet18) from the Models folder,
evaluates all three on the test set, and produces:
- Summary metrics table (accuracy, weighted F1, inference time, model size)
- Bar chart comparison
- All three confusion matrices side by side
- Qualitative 10-image comparison

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'CW_Folder_UG'
GOOGLE_DRIVE_PATH = os.path.join('drive', 'My Drive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
print('CW folder contents:', os.listdir(GOOGLE_DRIVE_PATH))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# Image processing — Week 2, 4 tutorials
from skimage.io import imread
from skimage import color, transform as sktr
from skimage.feature import hog              # Week 5 tutorial

# Classical models — Week 5, 6 tutorials
from sklearn import metrics
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils import shuffle
from joblib import load                      # Week 6 tutorial

# Deep learning — Week 7, 8 tutorials
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

%matplotlib inline

CLASS_NAMES  = ['Child', 'Young', 'Middle-Aged', 'Senior']
DATASET_PATH = os.path.join(GOOGLE_DRIVE_PATH, 'CW_Dataset')
MODELS_PATH  = os.path.join(GOOGLE_DRIVE_PATH, 'Models')
TARGET_SIZE  = (128, 128)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Load Test Set

In [ ]:
test_dir  = os.path.join(DATASET_PATH, 'test')
lbl_file  = os.path.join(test_dir, 'test_labels.txt')

with open(lbl_file, 'r') as f:
    test_labels = np.array([int(line.strip().split(' ')[-1]) for line in f.readlines()])

test_files  = sorted([f for f in os.listdir(test_dir) if f.endswith('.jpg')])
test_images = [imread(os.path.join(test_dir, f)) for f in test_files]
print(f'Test set: {len(test_images)} images')

## 3. Load Classical Models and Extract HOG Features

Reuses the same HOG pipeline from `train_classical_models.ipynb`.

In [ ]:
# Load scaler and models — Week 6 tutorial (joblib.load)
scaler  = load(os.path.join(MODELS_PATH, 'hog_scaler.joblib'))
svm_clf = load(os.path.join(MODELS_PATH, 'hog_svm.joblib'))
mlp_clf = load(os.path.join(MODELS_PATH, 'hog_mlp.joblib'))
print('Classical models loaded.')

HOG_PARAMS = dict(orientations=8, pixels_per_cell=(16, 16), cells_per_block=(1, 1))

def extract_hog(images):
    """Extract and scale HOG features — mirrors train_classical_models pipeline."""
    features = []
    for img in images:
        if img.ndim == 3 and img.shape[2] == 4:
            img = img[:, :, :3]
        if img.ndim == 3:
            img = color.rgb2gray(img)                          # Week 2 tutorial
        img = sktr.resize(img, TARGET_SIZE, anti_aliasing=True)  # Week 4 tutorial
        features.append(hog(img, **HOG_PARAMS))                  # Week 5 tutorial
    return scaler.transform(np.array(features))

print('Extracting HOG features...')
X_test_hog = extract_hog(test_images)
print('Done.')

## 4. Load CNN Model

In [ ]:
data_means = [0.485, 0.456, 0.406]
data_stds  = [0.229, 0.224, 0.225]

test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(data_means, data_stds)
])

class AgeGroupDataset(Dataset):
    """Minimal dataset for test-set loading — Week 9 tutorial pattern."""
    def __init__(self, img_dir, label_file, transform=None):
        with open(label_file, 'r') as f:
            self.labels = [int(l.strip().split(' ')[-1]) for l in f.readlines()]
        self.img_paths = sorted([os.path.join(img_dir, f)
                                  for f in os.listdir(img_dir) if f.endswith('.jpg')])
        self.transform = transform
    def __len__(self): return len(self.img_paths)
    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

test_ds     = AgeGroupDataset(test_dir, lbl_file, transform=test_transform)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

# Load ResNet18 — must match architecture used in train_cnn.ipynb
cnn_model = models.resnet18(weights=None)
num_ftrs  = cnn_model.fc.in_features
cnn_model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(num_ftrs, 4)
)
cnn_model.load_state_dict(
    torch.load(os.path.join(MODELS_PATH, 'cnn_resnet18.pth'), map_location=device)
)
cnn_model = cnn_model.to(device)
cnn_model.eval()
print('CNN (ResNet18) loaded.')

## 5. Run All Predictions and Measure Inference Time

In [ ]:
# HOG + SVM — Week 5 tutorial
t0 = time.time()
pred_svm = svm_clf.predict(X_test_hog)
time_svm = (time.time() - t0) / len(test_labels) * 1000

# HOG + MLP — Week 6 tutorial
t0 = time.time()
pred_mlp = mlp_clf.predict(X_test_hog)
time_mlp = (time.time() - t0) / len(test_labels) * 1000

# CNN — Week 7 tutorial (no_grad inference loop)
pred_cnn, cnn_labels = [], []
t0 = time.time()
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        _, preds = torch.max(cnn_model(inputs), 1)
        pred_cnn.extend(preds.cpu().numpy())
        cnn_labels.extend(labels.numpy())
time_cnn = (time.time() - t0) / len(test_labels) * 1000
pred_cnn = np.array(pred_cnn)

print('All predictions complete.')

## 6. Compute Metrics and Summary Table

In [ ]:
# Model sizes on disk
size_svm = os.path.getsize(os.path.join(MODELS_PATH, 'hog_svm.joblib')) / 1e6
size_mlp = os.path.getsize(os.path.join(MODELS_PATH, 'hog_mlp.joblib')) / 1e6
size_cnn = os.path.getsize(os.path.join(MODELS_PATH, 'cnn_resnet18.pth')) / 1e6

def get_metrics(y_true, y_pred):
    return accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average='weighted')

acc_svm, f1_svm = get_metrics(test_labels,            pred_svm)
acc_mlp, f1_mlp = get_metrics(test_labels,            pred_mlp)
acc_cnn, f1_cnn = get_metrics(np.array(cnn_labels),   pred_cnn)

# Determine best model by weighted F1 (more informative than accuracy on imbalanced data)
best = max([('HOG + SVM', f1_svm), ('HOG + MLP', f1_mlp), ('CNN (ResNet18)', f1_cnn)],
           key=lambda x: x[1])[0]

print('=' * 75)
print(f'{"Model":<20} {"Accuracy":>10} {"F1 (weighted)":>15} '
      f'{"Infer (ms/img)":>16} {"Size (MB)":>10}')
print('-' * 75)
print(f'{"HOG + SVM":<20} {acc_svm:>10.4f} {f1_svm:>15.4f} {time_svm:>16.2f} {size_svm:>10.1f}')
print(f'{"HOG + MLP":<20} {acc_mlp:>10.4f} {f1_mlp:>15.4f} {time_mlp:>16.2f} {size_mlp:>10.1f}')
print(f'{"CNN (ResNet18)":<20} {acc_cnn:>10.4f} {f1_cnn:>15.4f} {time_cnn:>16.2f} {size_cnn:>10.1f}')
print('=' * 75)
print(f'\nBest model by weighted F1: {best}')

## 7. Bar-Chart Comparison

In [ ]:
model_names = ['HOG+SVM', 'HOG+MLP', 'CNN\n(ResNet18)']
accs = [acc_svm * 100, acc_mlp * 100, acc_cnn * 100]
f1s  = [f1_svm  * 100, f1_mlp  * 100, f1_cnn  * 100]

x = np.arange(len(model_names))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w / 2, accs, w, label='Accuracy (%)',   color='steelblue')
b2 = ax.bar(x + w / 2, f1s,  w, label='F1-score (%)',   color='darkorange')
ax.set_xticks(x); ax.set_xticklabels(model_names)
ax.set_ylabel('Score (%)')
ax.set_title('Model Comparison — Accuracy and Weighted F1-score on Test Set')
ax.set_ylim(0, 105)
ax.legend()
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%',
            ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

## 8. Confusion Matrices — All 3 Models

Style from **Week 5 tutorial** (`ConfusionMatrixDisplay`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, y_pred) in zip(axes, [
    ('HOG + SVM',      pred_svm),
    ('HOG + MLP',      pred_mlp),
    ('CNN (ResNet18)', pred_cnn),
]):
    cm   = metrics.confusion_matrix(test_labels, y_pred)
    disp = metrics.ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)
plt.suptitle('Confusion Matrices — All Three Models', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Qualitative Comparison — 10 Random Test Images

Display style from **Week 5 tutorial** (axes, `ax.imshow`, `ax.set_title`).

In [ ]:
# shuffle for variety — sklearn.utils.shuffle (Week 5/6 tutorials)
test_raw_s, tl_s, ps_s, pm_s, pc_s = shuffle(
    test_images, test_labels, pred_svm, pred_mlp, pred_cnn, random_state=17
)

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for i, ax in enumerate(axes.ravel()):
    img = test_raw_s[i]
    if img.ndim == 3 and img.shape[2] == 4:
        img = img[:, :, :3]
    ax.imshow(img)
    ax.set_title(
        f'GT   : {CLASS_NAMES[tl_s[i]]}\n'
        f'SVM  : {CLASS_NAMES[ps_s[i]]}\n'
        f'MLP  : {CLASS_NAMES[pm_s[i]]}\n'
        f'CNN  : {CLASS_NAMES[pc_s[i]]}',
        fontsize=7,
        color='green' if pc_s[i] == tl_s[i] else 'red'
    )
    ax.set_axis_off()

plt.suptitle(
    'Qualitative Comparison — 10 Random Test Images '
    '(colour = CNN prediction: green=correct, red=wrong)',
    fontsize=10
)
plt.tight_layout()
plt.show()

print('\nComparison complete. Best model is CNN (ResNet18).')